In [1]:
import sys
sys.path.append("..")

from src.data.load_data import load_articles, load_history, load_behaviors

DATA_ROOT = "../data/raw/ebnerd_large"

articles = load_articles(DATA_ROOT)
history = load_history(DATA_ROOT, split="train")
behaviors = load_behaviors(DATA_ROOT, split="validation")

print(articles.shape)
print(history.shape)
print(behaviors.shape)
print(history.columns)
print(behaviors.columns)

(125541, 21)
(788090, 5)
(12566385, 17)
Index(['user_id', 'impression_time_fixed', 'scroll_percentage_fixed',
       'article_id_fixed', 'read_time_fixed'],
      dtype='object')
Index(['impression_id', 'article_id', 'impression_time', 'read_time',
       'scroll_percentage', 'device_type', 'article_ids_inview',
       'article_ids_clicked', 'user_id', 'is_sso_user', 'gender', 'postcode',
       'age', 'is_subscriber', 'session_id', 'next_read_time',
       'next_scroll_percentage'],
      dtype='object')


In [2]:
print(behaviors.columns)
print(history.columns)

Index(['impression_id', 'article_id', 'impression_time', 'read_time',
       'scroll_percentage', 'device_type', 'article_ids_inview',
       'article_ids_clicked', 'user_id', 'is_sso_user', 'gender', 'postcode',
       'age', 'is_subscriber', 'session_id', 'next_read_time',
       'next_scroll_percentage'],
      dtype='object')
Index(['user_id', 'impression_time_fixed', 'scroll_percentage_fixed',
       'article_id_fixed', 'read_time_fixed'],
      dtype='object')


In [3]:
from src.models.content_based import (
    build_article_text,
    fit_vectorizer,
    build_article_id_to_index
)

article_text_df = build_article_text(articles)

vectorizer, article_matrix = fit_vectorizer(article_text_df)

article_id_to_idx = build_article_id_to_index(article_text_df)

print(article_text_df.head())
print(article_text_df.shape)
print(article_matrix.shape)

   article_id                                       article_text
0     3000022  Hanks beskyldt for mishandling Tom Hanks har a...
1     3000063  Bostrups aske spredt i Furesøen Studieværten b...
2     3000613  Jesper Olsen ramt af hjerneblødning Den tidlig...
3     3000700  Madonna topløs med heste 47-årige Madonna pose...
4     3000840  Otto Brandenburg er død Sangeren og skuespille...
(125541, 2)
(125541, 5000)


In [4]:
from src.models.content_based import build_user_profile
test_user_id = history["user_id"].iloc[0]
user_vector = build_user_profile(test_user_id, history, article_matrix, article_id_to_idx)
print(test_user_id)
print(user_vector.shape)

10029
(1, 5000)


In [5]:
from src.models.content_based import rank_candidates

test_row = behaviors.iloc[0]

test_user_id = test_row["user_id"]
candidate_ids = test_row["article_ids_inview"]

user_vector = build_user_profile(test_user_id, history, article_matrix, article_id_to_idx)
ranked_articles = rank_candidates(user_vector, candidate_ids, article_matrix, article_id_to_idx)

print("User:", test_user_id)
print("Candidates:", candidate_ids[:11])
print("Top ranked:", ranked_articles[:10])
print("Clicked:", test_row["article_ids_clicked"])

User: 21814
Candidates: [9230405 9784793 9784803 9784275 9782726 9783865 9784702 9782884 9783800
 9779807]
Top ranked: [9230405, 9782884, 9784275, 9783865, 9779807, 9782726, 9784803, 9784702, 9784793, 9783800]
Clicked: [9782884]


In [6]:
print(len(candidate_ids))
print(9782884 in candidate_ids)

10
True


In [7]:
from src.evaluation.evaluate_cb import evaluate_on_behaviors

results = evaluate_on_behaviors(
    behaviors_df=behaviors,
    history_df=history,
    article_matrix=article_matrix,
    article_id_to_idx=article_id_to_idx,
    k=10,
    max_rows=100
)

print(results)

{'precision@10': 0.09058823529411751, 'recall@10': 0.9058823529411765, 'n_events': 85}


In [8]:
behaviors["article_ids_clicked"].apply(len).value_counts()

article_ids_clicked
1     12494779
2        65626
3         3500
4         1083
5          602
6          362
7          201
8          135
9           68
10          21
11           6
12           2
Name: count, dtype: int64